In [1]:
#Add path to recording here
RecordingPath = r'\\goeppert\Vol2\Rolf\FreelyMovingEphys\2024\082324\J705'




In [2]:
import pandas as pd
import numpy as np

import statistics as stat 
import math 
from pylab import *
import os
from tqdm import tqdm

###############################################################################################################
#Creating save folder

SavePath =  os.path.join(RecordingPath,'Analysis') 

if not os.path.exists(SavePath):
    os.makedirs(SavePath)

###############################################################################################################
###Finding FM ephys files before DOI 
print('Finding FM ephys files before DOI')

PreFM_Path = RecordingPath + '/fm1'
Items = os.listdir(PreFM_Path)
for names in Items:
    if names.endswith("ephys_props.h5"):
        PreDOI_InputData = PreFM_Path + '/' + names
        
#Finding FM ephys files after DOI         
PostFM_Path = RecordingPath + '/fm2'           
Items = os.listdir(PostFM_Path)
for names in Items:
    if names.endswith("ephys_props.h5"):
        PostDOI_InputData = PostFM_Path + '/' + names


PreDOI_Data = pd.read_hdf(PreDOI_InputData)
PostDOI_Data = pd.read_hdf(PostDOI_InputData)

###############################################################################################################
### Single Unit Indexes from Step 1

Items = os.listdir(RecordingPath)
for names in Items:
    if names.endswith("SingleUnits_NoThresh.npy"):
        SingleUnits_File = RecordingPath + '/' + names

Single_Units = np.load(SingleUnits_File)

###############################################################################################################
###Calculating and Subtracting Baseline Values from Each PSTH
print('Calculating and Subtracting Baseline Values from Each PSTH')

data = []

for ii in tqdm(list(range(0,len(Single_Units)))):
    #Cell = GoodClusters[GazeShift_ResponsiveClusters[ii]]
    Cell = Single_Units[ii]
   
    #Pre DOI Left Gaze Shifts
    temp = PreDOI_Data.loc[Cell,'FmLt_gazeshift_leftPSTH'] 
    PreLeft_Baseline = stat.mean(PreDOI_Data.loc[Cell,'FmLt_gazeshift_leftPSTH'][:800])
    PreLeft = temp-PreLeft_Baseline
    PreLeft_temp = PreLeft[500:1500]
    temp_PreLeft_Abs = np.abs(PreLeft_temp)
    PreLeft_MM_Idx = temp_PreLeft_Abs.argmax()
    PreLeft_MM = PreLeft_temp[PreLeft_MM_Idx]

    #Pre DOI Right Gaze Shifts
    temp = PreDOI_Data.loc[Cell,'FmLt_gazeshift_rightPSTH'] 
    PreRight_Baseline = stat.mean(PreDOI_Data.loc[Cell,'FmLt_gazeshift_rightPSTH'][:800])
    PreRight = temp-PreRight_Baseline
    PreRight_temp = PreRight[500:1500]
    temp_PreRight_Abs = np.abs(PreRight_temp)
    PreRight_MM_Idx = temp_PreRight_Abs.argmax()
    PreRight_MM = PreRight_temp[PreRight_MM_Idx]

    #Post DOI Left Gaze Shifts
    temp = PostDOI_Data.loc[Cell,'FmLt_gazeshift_leftPSTH'] 
    PostLeft_Baseline = stat.mean(PostDOI_Data.loc[Cell,'FmLt_gazeshift_leftPSTH'][:800])
    PostLeft = temp-PostLeft_Baseline
    PostLeft_temp = PostLeft[500:1500]
    temp_PostLeft_Abs = np.abs(PostLeft_temp)
    PostLeft_MM_Idx = temp_PostLeft_Abs.argmax()
    PostLeft_MM = PostLeft_temp[PostLeft_MM_Idx]

    #Post DOI Right Gaze Shifts
    temp = PostDOI_Data.loc[Cell,'FmLt_gazeshift_rightPSTH'] 
    PostRight_Baseline = stat.mean(PostDOI_Data.loc[Cell,'FmLt_gazeshift_rightPSTH'][:800])
    PostRight = temp-PostRight_Baseline
    PostRight_temp = PostRight[500:1500]
    temp_PostRight_Abs = np.abs(PostRight_temp)
    PostRight_MM_Idx = temp_PostRight_Abs.argmax()
    PostRight_MM = PostRight_temp[PostRight_MM_Idx]

    Baselines = [PreLeft_Baseline, PreRight_Baseline, PostLeft_Baseline, PostRight_Baseline]
    
    Pre_GSDS = (np.abs(PreLeft_MM) - np.abs(PreRight_MM))/(np.abs(PreLeft_MM) + np.abs(PreRight_MM))
    Post_GSDS = (np.abs(PostLeft_MM) - np.abs(PostRight_MM))/(np.abs(PostLeft_MM) + np.abs(PostRight_MM))

    data.append({'Cell': Cell, 'Baselines': Baselines, \
                'PreLeft': PreLeft, 'PreLeft_MM': PreLeft_MM, 'PreLeft_t': PreLeft_MM_Idx, \
                'PreRight': PreRight, 'PreRight_MM': PreRight_MM, 'PreRight_t': PreRight_MM_Idx, \
                'PostLeft': PostLeft, 'PostLeft_MM': PostLeft_MM, 'PostLeft_t': PostLeft_MM_Idx, \
                'PostRight': PostRight, 'PostRight_MM': PostRight_MM, 'PostRight_t': PostRight_MM_Idx, \
                'Pre_GSDS': Pre_GSDS, 'Post_GSDS': Post_GSDS})

FMData = pd.DataFrame(data)

###############################################################################################################
###Selecting the preferred Gaze Shift Response before and after DOI for future analysis - At this point I am not forcing the same cell to have the same direction preference before or after DOI
print('Selecting the preferred Gaze Shift Response before and after DOI')

Pref_Gaze_Pre = []
Pref_GazeMM_Pre = []
Pref_GazeT_Pre = []
Dir_Pref_Pre = []

Pref_Gaze_Post = []
Pref_GazeMM_Post = []
Pref_GazeT_Post = []
Dir_Pref_Post = []


for ii in tqdm(list(range(0,len(FMData)))):

    if FMData.loc[ii,'Pre_GSDS'] > 0:
        Dir_Pref_Pre.append('L')
        Pref_Gaze_Pre.append(FMData.loc[ii,'PreLeft'])
        Pref_GazeMM_Pre.append(FMData.loc[ii,'PreLeft_MM'])
        Pref_GazeT_Pre.append(FMData.loc[ii,'PreLeft_t'])
    else:
        Dir_Pref_Pre.append('R')
        Pref_Gaze_Pre.append(FMData.loc[ii,'PreRight'])
        Pref_GazeMM_Pre.append(FMData.loc[ii,'PreRight_MM'])
        Pref_GazeT_Pre.append(FMData.loc[ii,'PreRight_t'])
    

    if FMData.loc[ii,'Post_GSDS'] > 0:
        Dir_Pref_Post.append('L')
        Pref_Gaze_Post.append(FMData.loc[ii,'PostLeft'])
        Pref_GazeMM_Post.append(FMData.loc[ii,'PostLeft_MM'])
        Pref_GazeT_Post.append(FMData.loc[ii,'PostLeft_t'])
    else:
        Dir_Pref_Post.append('R')
        Pref_Gaze_Post.append(FMData.loc[ii,'PostRight'])
        Pref_GazeMM_Post.append(FMData.loc[ii,'PostRight_MM'])
        Pref_GazeT_Post.append(FMData.loc[ii,'PostRight_t'])

        
#Adding the Pref_Gaze Shift (Pre and Post) to the dataframe
FMData['Dir_Pref_Pre'] = Dir_Pref_Pre
FMData['Pref_Gaze_Pre'] = Pref_Gaze_Pre
FMData['Pref_GazeMM_Pre'] = Pref_GazeMM_Pre
FMData['Pref_GazeT_Pre'] = Pref_GazeT_Pre

FMData['Dir_Pref_Post'] = Dir_Pref_Post
FMData['Pref_Gaze_Post'] = Pref_Gaze_Post
FMData['Pref_GazeMM_Post'] = Pref_GazeMM_Post
FMData['Pref_GazeT_Post'] = Pref_GazeT_Post

###############################################################################################################
###Normalizing Gaze Shifts to Max Response
print('Normalizing Gaze Shifts to Max Response')

Pref_Gaze_Pre_Norm = []
Pref_Gaze_Post_Norm = []

for ii in tqdm(list(range(0,len(FMData)))):
    temp = np.nan_to_num(np.concatenate((FMData.loc[ii,'Pref_Gaze_Pre'], FMData.loc[ii,'Pref_Gaze_Post'])))
    temp = np.abs(temp)

    Pref_Gaze_Pre_Norm.append(np.nan_to_num(FMData.loc[ii,'Pref_Gaze_Pre'])/np.max(temp))
    Pref_Gaze_Post_Norm.append(np.nan_to_num(FMData.loc[ii,'Pref_Gaze_Post'])/np.max(temp))

FMData['Pref_Gaze_Pre_Norm'] = Pref_Gaze_Pre_Norm
FMData['Pref_Gaze_Post_Norm'] = Pref_Gaze_Post_Norm

###############################################################################################################
### Calculating Cluster type
print('Calculating cluster type')

import fmEphys as fme
import pickle


Pref_Gaze_Pre_Norm = np.stack(FMData['Pref_Gaze_Pre_Norm']) 
Pref_Gaze_Post_Norm = np.stack(FMData['Pref_Gaze_Post_Norm'])

# Specify the path to the KMeans model pickle file
km_model_path = r'//goeppert/nlab-nas/Shelby/SC_analysis/KMeans_PSTH_model_062022.pickle'
pca_model_path = r'//goeppert/nlab-nas/Shelby/SC_analysis/PCA_PSTH_model_062022.pickle'
with open(km_model_path, 'rb') as f:
    kmeans_model = pickle.load(f)
with open(pca_model_path, 'rb') as f:
    pca = pickle.load(f)

# Transform into PC space.
proj = pca.transform(Pref_Gaze_Pre_Norm[:, 950:1300])
projpost = pca.transform(Pref_Gaze_Post_Norm[:, 950:1300])

# Only keep required PCs
gproj = proj[:,:4]
gprojpost = projpost[:,:4]

# Map onto k-means clusters.
labels = kmeans_model.predict(gproj)
labelspost = kmeans_model.predict(gprojpost)

FMData['KMeans_Cluster_Pre'] = labels
FMData['KMeans_Cluster_Post'] = labelspost 

###############################################################################################################
#Saving Data into Mouse Recording Folder
print('Saving Data into Mouse Analysis Folder')
FMData.to_hdf(os.path.join(os.path.join(SavePath,'FM_data.h5')), 'w')
print('Done!')

Finding FM ephys files before DOI
Calculating and Subtracting Baseline Values from Each PSTH


100%|██████████| 158/158 [00:00<00:00, 158.03it/s]


Selecting the preferred Gaze Shift Response before and after DOI


100%|██████████| 158/158 [00:00<00:00, 13201.98it/s]


Normalizing Gaze Shifts to Max Response


100%|██████████| 158/158 [00:00<00:00, 4062.23it/s]


Calculating cluster type
Loading DLC 2.3.0...
Saving Data into Mouse Analysis Folder
Done!
